# Lab 3: Measure tank fill level and volume from imagery

**Course:** Politics of World Energy Rivalry (Stanford, Policy)

**What you will do**

1. Learn how a floating roof and the sun's angle reveal how full a tank is.
2. Test the method on simulated images where you know the right answer.
3. Find out how pixel size, sun angle and season limit accuracy at each site.
4. Detect tanks directly from an image (for sites with poor OSM coverage).
5. Run the method on real imagery from Labs 1 and 2.

**Why this matters for policy:** Fill levels are an independent check on claims about refinery output, stockpiles and sanctions. A method is only useful if you know its error. This lab teaches you to measure that error before you trust a number.

## Concepts you need first

**Floating roof tank.** A tall steel cylinder with a lid that floats on the oil. When oil goes in, the lid rises. When oil comes out, the lid sinks.

**Why shadows show the fill level.** The tank wall sticks up above the lid. When the lid is low, the wall casts a shadow onto it. The lower the lid, the bigger the shadow.

Two shadows matter:

- **Outer shadow:** the wall's shadow on the ground, outside the tank. Its length shows the **full height** of the tank wall.
- **Inner shadow:** the wall's shadow on the floating roof, inside the tank. Its length shows **how far the roof sits below the rim**.

**The key formulas.** Let H be the wall height, d the depth of the roof below the rim, and e the sun's elevation angle.

- Outer shadow length: `L_out = H / tan(e)`
- Inner shadow length: `L_in = d / tan(e)`
- Fill fraction: `f = 1 - d / H = 1 - L_in / L_out`
- Volume: `V = f × π × r² × H`

The sun angle cancels out of the fill fraction. You only need the ratio of the two shadow lengths.

**Shadow direction.** Shadows fall opposite the sun. If the sun's azimuth is 135° (southeast), shadows point toward 315° (northwest).

**What a pixel can resolve.** One pixel of shadow width equals `pixel size × tan(e)` of roof depth. A higher sun gives a coarser measure. A bigger pixel gives a coarser measure.

**Sources of error.** Pixel size, blur, wrong sun angle, wrong tank radius, tanks that shade each other, and clouds.

In [ ]:
import json
import re
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, map_coordinates
from skimage.filters import threshold_multiotsu
from skimage.feature import canny
from skimage.transform import hough_circle, hough_circle_peaks

REPO = Path("..").resolve()
AOI_DIR = REPO / "AOI"
TANK_DIR = REPO / "data" / "tanks"
IMG_DIR = REPO / "data" / "imagery"

BARRELS_PER_M3 = 6.2898     # one cubic meter holds about 6.29 barrels

## Step 1: The geometry in code

These two small functions turn the formulas into code.

In [ ]:
def shadow_length(height_m, sun_elev_deg):
    """Length on flat ground of the shadow cast by something height_m tall."""
    return height_m / np.tan(np.radians(sun_elev_deg))


def depth_per_pixel(pixel_m, sun_elev_deg):
    """How much roof depth one pixel of shadow width stands for."""
    return pixel_m * np.tan(np.radians(sun_elev_deg))


# Example: a 15 m tank with the sun at 30 degrees.
print("Outer shadow:", round(shadow_length(15, 30), 1), "m")
print("One 3 m pixel covers", round(depth_per_pixel(3.0, 30), 2), "m of roof depth")
print("One 0.5 m pixel covers", round(depth_per_pixel(0.5, 30), 2), "m of roof depth")

## Step 2: A simulated tank

To test a method you need a case where you know the answer. The function below draws an overhead image of floating-roof tanks. You choose the fill level, tank size, sun angle and pixel size. It follows the same geometry as above, then blurs the image and adds noise to mimic a real sensor.

You do not need to read every line. Read the comments to see the idea. The part to remember: **the image is built from the true fill level, so we can check our measurement against it.**

In [ ]:
def render_scene(tanks, sun_az, sun_elev, pixel_m, half_extent_m,
                 hires_m=0.5, noise=0.03, blur_sigma_px=0.6, seed=0):
    """Draw a grayscale overhead image (north is up) of floating-roof tanks.

    tanks: list of dicts with x, y (meters from image center), diameter_m, height_m, fill (0 to 1)
    Returns the image as a 2D array of brightness values between 0 and 1.
    """
    rng = np.random.default_rng(seed)                       # reproducible noise
    k = int(round(pixel_m / hires_m))                       # fine cells per output pixel
    n_out = int(round(2 * half_extent_m / pixel_m))         # output pixels per side
    n_hi = n_out * k                                        # fine cells per side

    # Coordinates of every fine cell, in meters. Rows run north to south.
    coords = (np.arange(n_hi) + 0.5) * hires_m - half_extent_m
    X, Y = np.meshgrid(coords, -coords)

    # Ground: mid-gray with gentle texture.
    img = 0.35 + 0.02 * gaussian_filter(rng.standard_normal((n_hi, n_hi)), 6) * 6

    # Unit vector pointing the way shadows fall (east, north components).
    az_s = np.radians((sun_az + 180) % 360)
    sx, sy = np.sin(az_s), np.cos(az_s)
    tan_e = np.tan(np.radians(sun_elev))

    inside_any = np.zeros(img.shape, dtype=bool)
    ground_shadow = np.zeros(img.shape, dtype=bool)

    for tk in tanks:
        R = tk["diameter_m"] / 2
        H = tk["height_m"]
        L_out = H / tan_e                                    # outer shadow length
        dx, dy = X - tk["x"], Y - tk["y"]                    # offset from tank center

        # Outer shadow = all points within R of the line from the tank center
        # to a point L_out away in the shadow direction.
        along = np.clip(dx * sx + dy * sy, 0, L_out)
        dist = np.hypot(dx - along * sx, dy - along * sy)
        ground_shadow |= dist <= R

        inside = np.hypot(dx, dy) <= R                       # cells on the roof

        # Inner shadow: a roof cell is shaded if the wall rim blocks the sun.
        # t is the horizontal distance from the cell to the wall, toward the sun.
        d = (1 - tk["fill"]) * H                             # roof depth below rim
        ps = dx * sx + dy * sy
        t = ps + np.sqrt(np.maximum(ps ** 2 + R ** 2 - (dx ** 2 + dy ** 2), 0))
        roof_shaded = t < d / tan_e

        roof = np.where(roof_shaded, 0.12, 0.60)             # dark in shadow, bright in sun
        img[inside] = roof[inside]
        inside_any |= inside

    img[ground_shadow & ~inside_any] = 0.09                  # dark outer shadow on ground

    # Mimic a real sensor: blur, then average fine cells into pixels, then add noise.
    img = gaussian_filter(img, blur_sigma_px * pixel_m / hires_m)
    img = img.reshape(n_out, k, n_out, k).mean(axis=(1, 3))
    img = img + rng.normal(0, noise, img.shape)
    return np.clip(img, 0, 1)


def to_pixel(x_m, y_m, pixel_m, half_extent_m):
    """Convert meters from image center to (column, row) pixel coordinates."""
    col = (x_m + half_extent_m) / pixel_m - 0.5
    row = (half_extent_m - y_m) / pixel_m - 0.5
    return col, row

In [ ]:
# One tank: 60 m wide, 15 m tall, 40 percent full. Sun in the southeast at 30 degrees.
tank = {"x": 0, "y": 0, "diameter_m": 60, "height_m": 15, "fill": 0.40}
SUN_AZ, SUN_EL = 135, 30
HALF = 120   # image covers 240 m by 240 m

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, pix in zip(axes, [3.0, 0.5]):
    img = render_scene([tank], SUN_AZ, SUN_EL, pix, HALF)
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{pix} m pixels ({img.shape[0]} x {img.shape[1]})")
    ax.axis("off")
plt.suptitle("Same tank, two sensors: PlanetScope (left) and SkySat (right)")
plt.show()

The bright disk is the floating roof. The dark crescent on the southeast edge is the **inner shadow**. The dark band running northwest of the tank is the **outer shadow**.

## Step 3: Measure the shadows

The measuring function does five things:

1. Find the sun-shadow direction from the sun azimuth.
2. Read the brightness along a line through the tank center in that direction (the **profile**).
3. Pick a brightness threshold that separates shadow from everything else (**multi-Otsu thresholding**, which finds natural breaks in the brightness values).
4. Measure the dark run at the start of the line (inner shadow) and the dark run past the far wall (outer shadow).
5. Compute fill = 1 - inner ÷ outer, then volume.

It also returns **flags** that warn you when a result is shaky.

In [ ]:
def measure_fill(img, col, row, radius_px, sun_az, sun_elev, pixel_m, height_m=None):
    """Estimate fill fraction and volume for one tank in one image.

    col, row: tank center in pixel coordinates; radius_px: tank radius in pixels.
    height_m: wall height if known. If None, it is estimated from the outer shadow.
    """
    # 1. Choose a brightness threshold from a square chip around the tank.
    r = int(5 * radius_px)
    r0, r1 = max(int(row) - r, 0), min(int(row) + r, img.shape[0])
    c0, c1 = max(int(col) - r, 0), min(int(col) + r, img.shape[1])
    chip = img[r0:r1, c0:c1]
    thr = threshold_multiotsu(chip, classes=3)[0]     # lowest break = shadow vs the rest

    # 2. Sample brightness along the shadow direction through the tank center.
    az_s = np.radians((sun_az + 180) % 360)
    dcol, drow = np.sin(az_s), -np.cos(az_s)          # image rows grow southward
    step = 0.1                                        # sample every 0.1 pixel
    t = np.arange(-1.5 * radius_px, 8 * radius_px, step)    # pixels from tank center
    smooth = gaussian_filter(img, 0.5)
    prof = map_coordinates(smooth, [row + t * drow, col + t * dcol], order=1, mode="nearest")
    dark = prof < thr

    def dark_run_end(start_t, tol=1.0):
        """Length of the dark run that begins within tol pixels of start_t."""
        near = np.where((np.abs(t - start_t) <= tol) & dark)[0]
        if len(near) == 0:
            return 0.0                                # no shadow found there
        i = near[0]
        while i < len(t) - 1 and dark[i + 1]:         # walk to the end of the run
            i += 1
        if i >= len(t) - 1:
            return t[-1] - start_t
        # Linear interpolation for a sub-pixel edge position.
        frac = (thr - prof[i]) / (prof[i + 1] - prof[i] + 1e-9)
        return (t[i] + frac * step) - start_t

    L_in_px = dark_run_end(-radius_px)                # starts at the sun-side wall
    L_out_px = dark_run_end(+radius_px)               # starts at the far wall

    flags = []
    if L_out_px < 2:
        flags.append("outer_shadow_too_short")
    if L_in_px >= 2 * radius_px - 1:
        flags.append("roof_fully_shaded_fill_is_upper_bound")
        L_in_px = 2 * radius_px
    if L_out_px > 7 * radius_px:
        flags.append("outer_shadow_runs_off_search_range")

    if L_out_px < 2:                                  # cannot compute a ratio
        return {"fill": np.nan, "flags": ";".join(flags), "threshold": thr,
                "t": t, "profile": prof}

    fill = float(np.clip(1 - L_in_px / L_out_px, 0, 1))
    L_in_m, L_out_m = L_in_px * pixel_m, L_out_px * pixel_m
    H_est = L_out_m * np.tan(np.radians(sun_elev))    # wall height from outer shadow
    H_used = height_m if height_m else H_est
    radius_m = radius_px * pixel_m
    volume_m3 = fill * np.pi * radius_m ** 2 * H_used

    # Rough error: one pixel of uncertainty on each shadow edge.
    fill_err = (1.0 / L_out_px) * np.sqrt(1 + (L_in_px / L_out_px) ** 2)

    return {"fill": fill, "fill_err_1px": fill_err, "L_in_m": L_in_m, "L_out_m": L_out_m,
            "height_est_m": H_est, "volume_m3": volume_m3,
            "barrels": volume_m3 * BARRELS_PER_M3,
            "flags": ";".join(flags), "threshold": thr, "t": t, "profile": prof}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for j, pix in enumerate([3.0, 0.5]):
    img = render_scene([tank], SUN_AZ, SUN_EL, pix, HALF)
    col, row = to_pixel(0, 0, pix, HALF)
    res = measure_fill(img, col, row, tank["diameter_m"] / 2 / pix, SUN_AZ, SUN_EL, pix,
                       height_m=tank["height_m"])

    axes[0, j].imshow(img, cmap="gray", vmin=0, vmax=1)
    # Draw the line we sampled along.
    az_s = np.radians((SUN_AZ + 180) % 360)
    tt = res["t"]
    axes[0, j].plot(col + tt * np.sin(az_s), row - tt * np.cos(az_s), color="orange", linewidth=1)
    axes[0, j].set_title(f"{pix} m pixels: measured fill {res['fill']:.2f} (true 0.40)")
    axes[0, j].axis("off")

    axes[1, j].plot(res["t"] * pix, res["profile"])
    axes[1, j].axhline(res["threshold"], color="red", linestyle="--", label="shadow threshold")
    axes[1, j].axvline(-tank["diameter_m"] / 2, color="gray", linestyle=":")
    axes[1, j].axvline(+tank["diameter_m"] / 2, color="gray", linestyle=":")
    axes[1, j].set_xlabel("Distance from tank center along shadow direction (m)")
    axes[1, j].set_ylabel("Brightness")
    axes[1, j].legend()
plt.tight_layout()
plt.show()

In the profile, the left dotted line is the sun-side wall and the right dotted line is the far wall. Brightness drops below the red line twice: once just inside the left wall (**inner shadow**) and once just outside the right wall (**outer shadow**). The ratio of the two dips gives the fill.

## Step 4: How accurate is it?

We now test the method many times. For each combination of fill level, sun elevation and pixel size we:

1. draw a tank at a random position within a pixel,
2. measure its fill,
3. compare with the true fill.

The summary statistic is **RMSE**, the root-mean-square error. Smaller is better. An RMSE of 0.05 means a typical error of five percentage points of fill.

In [ ]:
rng = np.random.default_rng(42)
fills = [0.1, 0.25, 0.4, 0.55, 0.7, 0.85]
sun_elevs = [15, 25, 35, 45, 60, 70]
pixels = [3.0, 0.5]
REPS = 4                       # random positions per case

records = []
for pix in pixels:
    for el in sun_elevs:
        for f in fills:
            for rep in range(REPS):
                # Random shift inside one pixel so edges fall differently each time.
                ox, oy = rng.uniform(-pix / 2, pix / 2, 2)
                tk = {"x": ox, "y": oy, "diameter_m": 60, "height_m": 15, "fill": f}
                img = render_scene([tk], SUN_AZ, el, pix, 150, seed=int(rng.integers(1e6)))
                col, row = to_pixel(ox, oy, pix, 150)
                res = measure_fill(img, col, row, 30 / pix, SUN_AZ, el, pix, height_m=15)
                records.append({"pixel_m": pix, "sun_elev": el, "true_fill": f,
                                "measured_fill": res["fill"], "flags": res["flags"]})

sweep = pd.DataFrame(records)
sweep["error"] = sweep["measured_fill"] - sweep["true_fill"]
print(len(sweep), "test cases;", int(sweep["measured_fill"].isna().sum()), "returned no answer")

In [ ]:
rmse = (sweep.dropna(subset=["error"])
        .groupby(["pixel_m", "sun_elev"])["error"]
        .apply(lambda e: np.sqrt((e ** 2).mean()))
        .unstack(0))

fig, ax = plt.subplots(figsize=(7, 4))
for pix in pixels:
    ax.plot(rmse.index, rmse[pix], marker="o", label=f"{pix} m pixels")
ax.set_xlabel("Sun elevation (degrees)")
ax.set_ylabel("RMSE of fill fraction")
ax.set_title("Measurement error vs. sun angle (60 m wide, 15 m tall tank)")
ax.legend()
plt.show()

# RMSE hides the direction of the error. The mean error shows bias:
# positive means the method reports tanks fuller than they are.
bias = (sweep.dropna(subset=["error"])
        .groupby(["pixel_m", "sun_elev"])["error"].mean().unstack(0))
print("RMSE of fill fraction")
print(rmse.round(3))
print("\nMean error (bias) of fill fraction")
print(bias.round(3))

**Reading the chart.** Error falls as the sun gets lower and rises as the sun gets higher. SkySat stays more accurate at every angle. At a high sun, 3 m pixels cannot see the inner shadow of a tank that is nearly full.

**Check the bias table too.** RMSE does not say which way the errors lean. If the mean error is mostly positive, the method reads tanks as fuller than they are. A biased method can look precise and still mislead, so always report both numbers.

Check the `flags` column to see where the method knew it was in trouble:

In [ ]:
sweep["flag_count"] = (sweep["flags"] != "").astype(int)   # 1 if any warning was raised
print(sweep.groupby(["pixel_m", "sun_elev"])["flag_count"].sum().unstack(0))

## Step 5: When can you measure at each site?

Planet's PlanetScope satellites cross the equator around 10:30 local solar time. The sun's elevation at that hour depends on latitude and date. The function below computes it, then the heatmap shows how much roof depth one 3 m pixel stands for in each month.

Lower numbers are better. A rule of thumb used here: pixel depth should be no more than about 20 percent of the tank height. For a 15 m tank that is 3 m.

In [ ]:
def sun_elevation_at(lat_deg, month, solar_hour=10.5):
    """Sun elevation (degrees) at a local solar hour, mid-month."""
    day_of_year = 15 + 30.4 * (month - 1)
    # Solar declination: the latitude where the sun is overhead on that date.
    decl = np.radians(23.44 * np.sin(np.radians(360 / 365 * (day_of_year - 81))))
    lat = np.radians(lat_deg)
    hour_angle = np.radians(15 * (solar_hour - 12))
    sin_el = np.sin(lat) * np.sin(decl) + np.cos(lat) * np.cos(decl) * np.cos(hour_angle)
    return np.degrees(np.arcsin(sin_el))


# Read each site's latitude from its AOI file.
site_lat = {}
for path in sorted(AOI_DIR.glob("*_aoi.geojson")):
    props = json.loads(path.read_text())["features"][0]["properties"]
    site_lat[props["facility"]] = props["center_lat"]

months = np.arange(1, 13)
table = pd.DataFrame(
    {name: [depth_per_pixel(3.0, sun_elevation_at(lat, m)) for m in months]
     for name, lat in site_lat.items()},
    index=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
).T

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(table.values, cmap="RdYlGn_r", vmin=0, vmax=8, aspect="auto")
ax.set_xticks(range(12)); ax.set_xticklabels(table.columns)
ax.set_yticks(range(len(table))); ax.set_yticklabels(table.index)
for i in range(table.shape[0]):
    for j in range(table.shape[1]):
        ax.text(j, i, f"{table.values[i, j]:.1f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="Roof depth per 3 m pixel (m)")
ax.set_title("PlanetScope roof-depth resolution by site and month (lower is better)")
plt.show()

## Step 6: Find tanks in an image

For sites where OSM has few tanks, we can look for circles in the image directly.

**How it works.** The Hough circle transform tests many possible circles and counts how many edge pixels line up with each one. Circles with many matching edge pixels are likely tanks. Canny edge detection first reduces the image to its outlines. A last check keeps only circles that are brighter inside than just outside.

We test it on a simulated image with four tanks.

In [ ]:
def ring_contrast(img, col, row, radius):
    """Mean brightness inside a circle minus mean brightness in a thin ring just outside it."""
    yy, xx = np.ogrid[:img.shape[0], :img.shape[1]]          # row and column index grids
    dist = np.hypot(xx - col, yy - row)                      # distance of every pixel from the center
    inside = img[dist <= 0.9 * radius].mean()
    ring = img[(dist > radius + 1) & (dist <= radius + 4)].mean()
    return inside - ring


def detect_tanks(img, pixel_m, min_d_m=25, max_d_m=100, min_contrast=0.10):
    """Find circular tanks in an image. Returns centers and radii in pixels."""
    edges = canny(img, sigma=1.5)                         # outline the bright/dark boundaries
    r_min = max(int(min_d_m / 2 / pixel_m), 3)
    r_max = int(max_d_m / 2 / pixel_m)
    radii = np.arange(r_min, r_max + 1)                   # radii to test, in pixels
    spaces = hough_circle(edges, radii)
    scores, cols, rows, rads = hough_circle_peaks(
        spaces, radii, min_xdistance=2, min_ydistance=2,
        threshold=0.6 * spaces.max(), total_num_peaks=100)

    # Keep the best-scoring circle in each place. Drop weaker circles that overlap it.
    kept = []
    for s_, c_, r_, rad_ in sorted(zip(scores, cols, rows, rads), key=lambda z: -z[0]):
        if all(np.hypot(c_ - k[1], r_ - k[2]) > 0.7 * max(rad_, k[3]) for k in kept):
            kept.append((s_, c_, r_, rad_))
    found = pd.DataFrame(kept, columns=["score", "col", "row", "radius_px"])

    # Shadow edges also look like circles. A real roof is brighter than the ground
    # around it, so drop circles that fail that test.
    found["contrast"] = [ring_contrast(img, r.col, r.row, r.radius_px) for r in found.itertuples()]
    return found[found["contrast"] >= min_contrast].reset_index(drop=True)


scene_tanks = [
    {"x": -70, "y": 60, "diameter_m": 70, "height_m": 18, "fill": 0.8},
    {"x": 40, "y": 70, "diameter_m": 55, "height_m": 15, "fill": 0.3},
    {"x": -50, "y": -60, "diameter_m": 60, "height_m": 15, "fill": 0.55},
    {"x": 70, "y": -50, "diameter_m": 80, "height_m": 20, "fill": 0.15},
]
HALF2, PIX2 = 180, 3.0
scene = render_scene(scene_tanks, SUN_AZ, 30, PIX2, HALF2, seed=3)
found = detect_tanks(scene, PIX2)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(scene, cmap="gray", vmin=0, vmax=1)
for _, f in found.iterrows():
    ax.add_patch(plt.Circle((f.col, f.row), f.radius_px, fill=False, color="red", linewidth=1.5))
ax.set_title(f"Detected {len(found)} circles (true: {len(scene_tanks)})")
ax.axis("off")
plt.show()
found

Compare the list with the true tanks. Two steps keep the output clean: overlapping circles are merged, and circles whose inside is not brighter than the ground around them are dropped (shadow edges often look like circles).

The method still fails in some cases. Change the sun elevation in the scene above from 30 to 20 and rerun. Tanks with mostly shaded roofs lose the brightness contrast and disappear. Detection output always needs a human check before measurement.

## Step 7: Run it on real imagery

This section uses the files from Labs 1 and 2. It reads a clipped GeoTIFF, finds each tank from the Lab 2 file (or from detection if none exists), looks up the sun angles in the Lab 1 catalog, and measures every tank.

**If you have no imagery yet,** the cells print a message and skip. Run Lab 1 with `DRY_RUN = False` first.

First, choose a site and a scene.

In [ ]:
import rasterio
from pyproj import Transformer

# Pick a site by its short name. Options are the AOI file names without "_aoi".
SITE = "sk_energy_ulsan_south_korea"

# Which bands to average into a brightness image.
# 8-band PlanetScope: 2 = blue, 4 = green, 6 = red. For 4-band files use (1, 2, 3).
BRIGHTNESS_BANDS = (2, 4, 6)

tif_candidates = sorted((IMG_DIR / SITE).glob("*clip*.tif")) if (IMG_DIR / SITE).exists() else []
tif_candidates = [p for p in tif_candidates if "udm" not in p.name.lower()]   # skip cloud masks
print(f"{len(tif_candidates)} image files found for {SITE}")
for p in tif_candidates:
    print("  ", p.name)

In [ ]:
HAVE_IMAGE = len(tif_candidates) > 0

if HAVE_IMAGE:
    # Choose the first image. Change the index to try another.
    tif_path = tif_candidates[0]

    # Match the file name to the catalog to get sun angles.
    catalog = pd.read_csv(IMG_DIR / f"catalog_{SITE}.csv")
    match = catalog[[tif_path.name.startswith(sid) for sid in catalog["id"]]]
    if match.empty:
        raise ValueError("Could not match this image to the catalog. Rerun Lab 1, Step 4.")
    scene_info = match.iloc[0]
    sun_az, sun_el = float(scene_info["sun_azimuth"]), float(scene_info["sun_elevation"])
    print("Scene:", scene_info["id"], "| sun azimuth", sun_az, "| elevation", sun_el)

    with rasterio.open(tif_path) as src:
        # Average the chosen bands to one brightness image and scale to 0 to 1.
        bands = src.read(BRIGHTNESS_BANDS).astype("float32")
        real_img = bands.mean(axis=0)
        lo, hi = np.percentile(real_img[real_img > 0], [2, 98])     # ignore blank edges
        real_img = np.clip((real_img - lo) / (hi - lo), 0, 1)
        transform, crs = src.transform, src.crs
        pixel_real = abs(transform.a)                                # pixel size in meters
        if abs(transform.b) > 1e-6 or abs(transform.d) > 1e-6:
            print("WARNING: image is rotated. This lab assumes north is up.")
    print("Image shape:", real_img.shape, "| pixel size:", round(pixel_real, 2), "m")

    plt.figure(figsize=(7, 7))
    plt.imshow(real_img, cmap="gray")
    plt.title(f"{SITE}: {scene_info['id']}")
    plt.axis("off")
    plt.show()
else:
    print("No imagery yet. Run Lab 1 with DRY_RUN = False, then come back.")

In [ ]:
if HAVE_IMAGE:
    tank_file = TANK_DIR / f"tanks_{SITE}.geojson"
    to_image_crs = Transformer.from_crs("EPSG:4326", crs, always_xy=True)   # lon/lat -> image CRS

    if tank_file.exists():
        # Use the Lab 2 inventory.
        import geopandas as gpd
        inv = gpd.read_file(tank_file)
        inv = inv[inv["diameter_m"] >= 8 * pixel_real]       # drop tanks too small to see
        xs, ys = to_image_crs.transform(inv["center_lon"].values, inv["center_lat"].values)
        cols, rows = ~transform * (xs, ys)                   # map coords -> pixel coords
        targets = pd.DataFrame({"col": cols - 0.5, "row": rows - 0.5,
                                "radius_px": inv["diameter_m"].values / 2 / pixel_real,
                                "source": "osm",
                                "osm_id": inv["osm_id"].values if "osm_id" in inv.columns else np.nan})
    else:
        print("No Lab 2 file for this site. Using image-based detection instead.")
        targets = detect_tanks(real_img, pixel_real)
        targets["source"] = "detected"
    print(len(targets), "tanks to measure")

In [ ]:
if HAVE_IMAGE:
    results = []
    for i, tk in targets.iterrows():
        res = measure_fill(real_img, tk["col"], tk["row"], tk["radius_px"],
                           sun_az, sun_el, pixel_real)         # height estimated from shadow
        results.append({"tank": i, "osm_id": tk.get("osm_id"), "source": tk["source"],
                        "diameter_m": 2 * tk["radius_px"] * pixel_real,
                        "fill": res.get("fill"), "fill_err_1px": res.get("fill_err_1px"),
                        "height_est_m": res.get("height_est_m"),
                        "barrels": res.get("barrels"), "flags": res["flags"]})
    metrics = pd.DataFrame(results)
    metrics.insert(0, "scene", scene_info["id"])
    out_csv = TANK_DIR / f"metrics_{SITE}_{scene_info['id']}.csv"
    metrics.to_csv(out_csv, index=False)

    good = metrics[(metrics["flags"] == "") & metrics["fill"].notna()]
    print(f"{len(good)} of {len(metrics)} tanks passed quality checks")
    print(f"Estimated volume in passing tanks: {good['barrels'].sum() / 1e6:.2f} million barrels")
    print("Saved", out_csv)
    metrics.round(2)

In [ ]:
if HAVE_IMAGE and len(metrics):
    # Contact sheet: each measured tank with its fill estimate.
    n = min(len(targets), 12)
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    for ax, (i, tk) in zip(axes.ravel(), targets.head(n).iterrows()):
        half = int(4 * tk["radius_px"])
        c, r = int(tk["col"]), int(tk["row"])
        chip = real_img[max(r - half, 0):r + half, max(c - half, 0):c + half]
        ax.imshow(chip, cmap="gray", vmin=0, vmax=1)
        fill = metrics.loc[i, "fill"]
        ax.set_title(f"Tank {i}: fill {fill:.2f}" if pd.notna(fill) else f"Tank {i}: no result",
                     fontsize=9)
        ax.axis("off")
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Check your work and think about error

Look at the contact sheet. For each tank, does the measured fill match what you see? Mark any tank where the shadow edge looks wrong.

Your numbers have at least these error sources:

- **Pixel size.** The `fill_err_1px` column shows the effect of a one-pixel mistake.
- **Sun angle.** A 2° error at 20° elevation changes the height estimate by about 10 percent.
- **Radius.** An OSM outline may be offset by a few meters from the tank in the image.
- **Tank type.** Fixed-roof and internal-floating-roof tanks show no usable shadow. The method will give a meaningless result for them. Look for a roof that appears flat, evenly bright and ringed by a visible rim.
- **Neighbors.** A nearby tank can shade the roof or the ground.

## Exercises

1. In Step 4, change the tank diameter to 40 m and then 100 m. How does RMSE change at 3 m pixels? What does that say about which tanks to trust?
2. In Step 5, which site and month would you pick for a one-day tasked SkySat pass? Which for free-running PlanetScope? Explain your choice in two sentences.
3. In Step 3, set `noise=0.10` in `render_scene`. What fails first, the threshold or the edge position? How would clouds or haze look to this method?
4. For your real-imagery run, take five tanks and measure their shadows by hand in QGIS. How far off is the code?
5. The mean error in Step 4 is almost as large as the RMSE, and it grows with sun elevation. Offer two possible causes (think about blur, the shadow threshold, and shadows shorter than one pixel). Then change `blur_sigma_px` in `render_scene` to 0.2 and 1.0 and rerun to test one of them.
6. Write a short memo (200 words) for a policymaker: what can and cannot be claimed from PlanetScope tank imagery at this site? Include one number with its error.

**Where this fits in the course:** The same logic applies to other open indicators of energy activity, such as flaring, ship tracking and night lights. In each case, the policy question is how much weight a noisy independent measurement should carry against an official statement.